In [8]:
import json
import os
from collections import defaultdict
from tqdm import tqdm

ANNOTATION_JSON = "dataset/info_test2020.json"
IMAGE_DIR = "dataset/images"
OUTPUT_JSONL = "fashionpedia_llm_input_1.jsonl"

with open(ANNOTATION_JSON, "r") as f:
    data = json.load(f)

image_map = {
    img["id"]: img["file_name"]
    for img in data["images"]
}

attribute_map = {
    attr["id"]: (
        attr["name"],
        attr["supercategory"]
    )
    for attr in data["attributes"]
}

image_attributes = defaultdict(list)
image_supercategories = defaultdict(set)

for ann in tqdm(data["annotations"], desc="Processing annotations"):
    img_id = ann["image_id"]

    for attr_id in ann.get("attribute_ids", []):
        if attr_id not in attribute_map:
            continue

        name, supercat = attribute_map[attr_id]

        image_attributes[img_id].append(name)
        image_supercategories[img_id].add(supercat)

with open(OUTPUT_JSONL, "w") as f:
    for img_id, filename in tqdm(
        image_map.items(),
        total=len(image_map),
        desc="Writing JSONL"
    ):
        sample = {
            "image_path": os.path.join(IMAGE_DIR, filename),
            "attributes": sorted(set(image_attributes.get(img_id, []))),
            "supercategories": sorted(image_supercategories.get(img_id, set()))
        }

        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(image_map)} samples to {OUTPUT_JSONL}")

KeyError: 'annotations'

In [9]:
import json
from collections import Counter

ANNOTATION_JSON = "dataset/attributes_val2020.json"

with open(ANNOTATION_JSON, "r") as f:
    data = json.load(f)

# All image_ids from annotations
image_ids = [ann["image_id"] for ann in data["annotations"]]

print(f"Total image_id entries: {len(image_ids)}")
print(f"Unique image_ids: {len(set(image_ids))}")

# Optional: statistics
counts = Counter(image_ids)

print(f"Images with multiple annotations: {sum(v > 1 for v in counts.values())}")
print(f"Maximum annotations for a single image: {max(counts.values())}")
print(f"Average annotations per image: {len(image_ids) / len(counts):.2f}")

Total image_id entries: 1158
Unique image_ids: 1158
Images with multiple annotations: 0
Maximum annotations for a single image: 1
Average annotations per image: 1.00
